## Environment: Python and PyTorch versions

In [ ]:
import sys, platform
import torch

print("Python version :", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("Platform       :", platform.platform())

Python version : 3.12.3
PyTorch version: 2.14.1+cu130
Platform       : Linux-6.18.44-fc-v70-x86_64-with-glibc2.39


## Activity 1: compute the predictions

The model is a linear model, y_hat = X @ w + b. First I run the code from the slide.

In [ ]:
X = torch.tensor([[1., 2.], [3., 4.], [5., 6.]])
w = torch.tensor([[0.5], [-0.25]])
b = torch.tensor(1.0)
y_hat = X @ w + b

print("Shapes:", X.shape, w.shape, y_hat.shape)
print(y_hat)

Shapes: torch.Size([3, 2]) torch.Size([2, 1]) torch.Size([3, 1])


tensor([[1.0000],
        [1.5000],
        [2.0000]])


The shapes are (3, 2), (2, 1) and (3, 1), which is the same as the expected shapes on the slide.

Manual check of the first prediction: y_hat_1 = 1(0.5) + 2(-0.25) + 1 = 0.5 - 0.5 + 1 = 1.0. For the other rows: y_hat_2 = 3(0.5) + 4(-0.25) + 1 = 1.5, and y_hat_3 = 5(0.5) + 6(-0.25) + 1 = 2.0. So I expect [1.0, 1.5, 2.0], and the code gives the same result.

### Predicted outputs alongside observed outputs

The slide does not provide observed values, so I use the targets y_obs = [1.0, 2.0, 3.0] as the observed outputs for the three samples.

In [ ]:
y_obs = torch.tensor([[1.0], [2.0], [3.0]])

print("sample | predicted (b=1.0) | observed | error (pred - obs)")
for i in range(3):
    p, o = y_hat[i].item(), y_obs[i].item()
    print(f"  {i+1}    |       {p:.2f}        |   {o:.2f}   |   {p - o:+.2f}")

sample | predicted (b=1.0) | observed | error (pred - obs)
  1    |       1.00        |   1.00   |   +0.00
  2    |       1.50        |   2.00   |   -0.50
  3    |       2.00        |   3.00   |   -1.00


## Matrix shapes and the broadcasting risk

**Matrix shapes.** X has shape (3, 2): 3 samples and 2 features. w has shape (2, 1): one weight for each feature. For the matrix product X @ w, the inner dimensions must match (2 and 2), and the result has the outer dimensions, (3, 1): one prediction for each sample. The bias b is a scalar (shape ()), so it is added to every prediction.

**Broadcasting risk.** Broadcasting automatically expands tensors with different shapes so that an element-wise operation can work. This is useful for adding the bias, but it can hide mistakes. If one tensor has shape (3,) and the other has shape (3, 1), PyTorch does not raise an error but expands them to (3, 3). Then, for example, an error calculation gives 9 numbers instead of 3 and the loss is wrong, without any warning. That is why I check the shapes of y_hat and y_obs before computing the loss. The demonstration below shows this.

In [ ]:
y_flat = torch.tensor([1.0, 2.0, 3.0])          # shape (3,)  <- wrong shape for the targets
print("y_hat shape :", y_hat.shape)
print("y_flat shape:", y_flat.shape)

diff = y_hat - y_flat                            # no error, but broadcasting to (3, 3)
print("y_hat - y_flat shape:", diff.shape)
print(diff)

wrong_mse = (diff ** 2).mean()
print("Wrong MSE (because of broadcasting):", wrong_mse.item())

# correct version: both tensors have shape (3, 1)
right_mse = ((y_hat - y_obs) ** 2).mean()
print("Correct MSE                        :", right_mse.item())

y_hat shape : torch.Size([3, 1])
y_flat shape: torch.Size([3])
y_hat - y_flat shape: torch.Size([3, 3])
tensor([[ 0.0000, -1.0000, -2.0000],
        [ 0.5000, -0.5000, -1.5000],
        [ 1.0000,  0.0000, -1.0000]])
Wrong MSE (because of broadcasting): 1.0833333730697632
Correct MSE                        : 0.4166666567325592


## Discussion: change b to 0.5

Before running the cell I predict the outputs. The weights part X @ w is [0, 0.5, 1.0], so with b = 0.5 the predictions should be **[0.5, 1.0, 1.5]**: each output is 0.5 lower than with b = 1.0.

In [ ]:
b2 = torch.tensor(0.5)
y_hat2 = X @ w + b2
print("Predictions with b = 0.5:")
print(y_hat2)
print("Matches my prediction [0.5, 1.0, 1.5]:", torch.allclose(y_hat2, torch.tensor([[0.5], [1.0], [1.5]])))

Predictions with b = 0.5:
tensor([[0.5000],
        [1.0000],
        [1.5000]])
Matches my prediction [0.5, 1.0, 1.5]: True


## MSE calculations for both bias values

MSE = mean of squared errors, (1/n) * sum((y_hat - y_obs)^2).

In [ ]:
def mse(pred, target):
    assert pred.shape == target.shape, f"shape mismatch: {pred.shape} vs {target.shape}"
    return ((pred - target) ** 2).mean()

mse1 = mse(y_hat, y_obs)
mse2 = mse(y_hat2, y_obs)

print("Observed          :", y_obs.flatten().tolist())
print("Predicted, b = 1.0:", y_hat.flatten().tolist(),  "-> MSE =", round(mse1.item(), 4))
print("Predicted, b = 0.5:", y_hat2.flatten().tolist(), "-> MSE =", round(mse2.item(), 4))

# manual check
print("Manual b=1.0:", ((0.0)**2 + (-0.5)**2 + (-1.0)**2) / 3)
print("Manual b=0.5:", (((-0.5)**2) + ((-1.0)**2) + ((-1.5)**2)) / 3)

Observed          : [1.0, 2.0, 3.0]
Predicted, b = 1.0: [1.0, 1.5, 2.0] -> MSE = 0.4167
Predicted, b = 0.5: [0.5, 1.0, 1.5] -> MSE = 1.1667
Manual b=1.0: 0.4166666666666667
Manual b=0.5: 1.1666666666666667


For b = 1.0 the errors are 0, -0.5 and -1.0, so MSE = (0 + 0.25 + 1.0) / 3 = 0.4167. For b = 0.5 the errors are -0.5, -1.0 and -1.5, so MSE = (0.25 + 1.0 + 2.25) / 3 = 1.1667. The model with b = 1.0 is better for these observed values, because the predictions are below the observed values and a smaller bias makes them even lower. The assert in the mse function protects against the shape problem described above.

## Activity 2: a simple reflex agent

The code from slide 36. The agent has two locations, A and B, and uses only the current location and the local dirt observation.

In [ ]:
def cleaning_agent(location, dirty):
    if dirty:
        return "clean"
    return "right" if location == "A" else "left"

for percept in [("A", True), ("A", False), ("B", False)]:
    print(percept, "->", cleaning_agent(*percept))

('A', True) -> clean
('A', False) -> right
('B', False) -> left


The output is clean, right, left, which is the expected result. The agent reacts only to the current percept and has no memory, so it moves between A and B forever, even when everything is already clean.

### Proposal: memory and a stopping rule if dirt never reappears

**Memory.** The agent needs to remember which locations it has already visited and found clean. A simple version is a dictionary, for example `{"A": "clean", "B": "unknown"}`, that is updated after every percept (a location is marked clean when the agent cleans it or sees that it is not dirty).

**Stopping condition.** The agent stops (returns "stop" or "no-op") when its memory says that all locations (A and B) are clean. This works only because of the assumption that dirt never reappears. If dirt could appear again, the agent would need to re-check the locations from time to time, for example after a fixed number of steps.

The code below is a small implementation of this idea.

In [ ]:
def make_memory_agent():
    memory = {"A": "unknown", "B": "unknown"}

    def agent(location, dirty):
        if dirty:
            memory[location] = "clean"      # after cleaning, the location is clean
            return "clean"
        memory[location] = "clean"          # observed as not dirty
        if all(v == "clean" for v in memory.values()):
            return "stop"                   # stopping rule
        return "right" if location == "A" else "left"

    return agent

agent = make_memory_agent()
for percept in [("A", True), ("A", False), ("B", False)]:
    print(percept, "->", agent(*percept))

('A', True) -> clean
('A', False) -> right
('B', False) -> stop


With memory the output is clean, right, stop. After the agent has checked both locations and knows that both are clean, it stops instead of moving forever.

## Summary

- Predictions with b = 1.0 are [1.0, 1.5, 2.0], and with b = 0.5 they are [0.5, 1.0, 1.5].
- MSE against the observed values: 0.4167 (b = 1.0) and 1.1667 (b = 0.5).
- Matching shapes ((3, 1) for both predictions and targets) are important, because broadcasting can silently give a wrong loss.
- A reflex agent without memory never stops; memory of clean locations plus a stopping rule fixes this when dirt never reappears.